In [19]:
import scanpy as sc
from openai import OpenAI
import json

In [3]:
adata = sc.datasets.pbmc3k_processed()
adata

AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None (.X)

In [6]:
# cell type annotation
adata.obs['louvain'].head()

index
AAACATACAACCAC-1        CD4 T cells
AAACATTGAGCTAC-1            B cells
AAACATTGATCAGC-1        CD4 T cells
AAACCGTGCTTCCG-1    CD14+ Monocytes
AAACCGTGTATGCG-1           NK cells
Name: louvain, dtype: category
Categories (8, str): ['CD4 T cells', 'CD14+ Monocytes', 'B cells', 'CD8 T cells',
                      'NK cells', 'FCGR3A+ Monocytes', 'Dendritic cells',
                      'Megakaryocytes']

In [ ]:
def get_dataset_info(adata, cell_type_key="louvain"):
    if cell_type_key not in adata.obs.columns:
        return {
            "error": f"column '{cell_type_key}' not found in adata.obs",
            "available_columns": adata.obs.columns.tolist(),
        }
    return {
        "n_cells": adata.n_obs,
        "n_genes": adata.n_vars,
        "cell_types": adata.obs[cell_type_key].astype(str).unique().tolist()
    }
print(get_dataset_info(adata))

{'n_cells': 2638, 'n_genes': 1838, 'cell_types': ['CD4 T cells', 'B cells', 'CD14+ Monocytes', 'NK cells', 'CD8 T cells', 'FCGR3A+ Monocytes', 'Dendritic cells', 'Megakaryocytes']}


In [ ]:
TOOLs = [
    {
        "type": "function",
        "name": "get_dataset_info",
        "description": (
            "Return the number of cells, genes, "
            "and available cell types in an scRNA-seq dataset"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "cell_type_key": {
                    "type": "string",
                    "description": "cell annotation column"
                }
            },
            "required": ["cell_type_key"],
            "additionalProperties": False
        },
        "strict": True
    }
]

In [16]:
client = OpenAI()
prompt = (
        "My cell type annotation column is 'louvain'. "
        "How many cells and genes are in my dataset?"
    )
response = client.responses.create(
    model = 'gpt-6-luna',
    input = prompt,
    tools = TOOLs,
    tool_choice = "auto"
)


In [17]:
for item in response.output:
    print(item)

ResponseFunctionToolCall(arguments='{"cell_type_key":"louvain"}', call_id='call_p1X4yU5v5uWudJ7AVkxuDCZs', name='get_dataset_info', type='function_call', id='fc_0efad3f70d1d2c6c006af918a3c83487d29c24fa44d5d921bf', async_=None, caller=None, namespace=None, status='completed')


In [18]:
response.output_text

''

In [ ]:
tool_outputs = []
for item in response.output:
    if item.type == "function_call" and item.name == "get_dataset_info":
        args = json.loads(item.arguments)
        try:
            result = get_dataset_info(adata, **args)
        except Exception as e:
            result = {"error": f"{type(e).__name__}: {e}"}
        tool_outputs.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result),
        })
tool_outputs


[{'type': 'function_call_output',
  'call_id': 'call_p1X4yU5v5uWudJ7AVkxuDCZs',
  'output': '{"n_cells": 2638, "n_genes": 1838, "cell_types": ["CD4 T cells", "B cells", "CD14+ Monocytes", "NK cells", "CD8 T cells", "FCGR3A+ Monocytes", "Dendritic cells", "Megakaryocytes"]}'}]

In [21]:
## send the results back so the model can write the answer
final = client.responses.create(
    model = 'gpt-6-luna',
    previous_response_id = response.id,
    input = tool_outputs,
    tools = TOOLs,
)
print(final.output_text)

Your dataset contains **2,638 cells** and **1,838 genes**.
